In [1]:
# Imports and API connection test
# Run once per session

import requests
import pandas as pd
import numpy as np
import torch
import pickle
import time
import warnings
warnings.filterwarnings('ignore')

from scipy.stats import norm as scipy_norm

# Your API key
ODDS_API_KEY = 'ac3f07151eb58b517d62cf941c79755a'

# Test connection — check WNBA is available
url      = 'https://api.the-odds-api.com/v4/sports'
response = requests.get(url, params={'apiKey': ODDS_API_KEY})

sports = response.json()
wnba   = [s for s in sports if 'wnba' in s['key'].lower()]

print(f"Status: {response.status_code}")
print()
print("WNBA markets available:")
for s in wnba:
    print(f"  {s['key']:<30} {s['title']}")

Status: 200

WNBA markets available:
  basketball_wnba                WNBA


In [3]:
# Load model and dataset
# Run once per session
import torch.nn as nn

# WNBA team abbreviation → full name mapping
team_name_map = {
    'ATL': 'Atlanta Dream', 'CHI': 'Chicago Sky', 'CON': 'Connecticut Sun',
    'DAL': 'Dallas Wings', 'GSV': 'Golden State Valkyries',
    'IND': 'Indiana Fever', 'LAS': 'Los Angeles Sparks', 'LVA': 'Las Vegas Aces',
    'MIN': 'Minnesota Lynx', 'NYL': 'New York Liberty',
    'PHO': 'Phoenix Mercury', 'PHX': 'Phoenix Mercury',
    'SEA': 'Seattle Storm', 'WAS': 'Washington Mystics',
    'PDX': 'Portland Fire', 'TOR': 'Toronto Tempo',
}
reverse_team_map = {v: k for k, v in team_name_map.items()}

CURRENT_SEASON = 2026

# Load WNBA dataset
df_master = pd.read_csv('wnba_master_dataset.csv', parse_dates=['GAME_DATE'])
df_master['HOME_AWAY'] = df_master['HOME_AWAY'].map({'HOME': 1, 'AWAY': 0})
df_master['POSITION']  = df_master['POSITION'].map({'G': 0, 'F': 1, 'C': 2})

# Fix roll10 NaNs for players with limited recent history
roll10_cols = ['PTS_roll10', 'REB_roll10', 'AST_roll10', 'BLK_roll10',
               'STL_roll10', 'FG3M_roll10', 'MIN_roll10', 'TOV_roll10',
               'FGA_roll10', 'FG3A_roll10']
roll5_cols  = ['PTS_roll5', 'REB_roll5', 'AST_roll5', 'BLK_roll5',
               'STL_roll5', 'FG3M_roll5', 'MIN_roll5', 'TOV_roll5',
               'FGA_roll5', 'FG3A_roll5']
for roll10, roll5 in zip(roll10_cols, roll5_cols):
    df_master[roll10] = df_master[roll10].fillna(df_master[roll5])

# Add position-normalized rolling features
pos_roll_cols = [
    'OPP_PTS_VS_POS_roll5', 'OPP_REB_VS_POS_roll5',
    'OPP_AST_VS_POS_roll5', 'OPP_BLK_VS_POS_roll5',
    'OPP_STL_VS_POS_roll5', 'OPP_3PM_VS_POS_roll5'
]
for col in pos_roll_cols:
    norm_col = f'{col}_norm'
    df_master[norm_col] = df_master.groupby('POSITION')[col].transform(
        lambda x: (x - x.mean()) / (x.std() + 1e-8)
    )

print(f"Dataset loaded: {len(df_master):,} rows")

# Load WNBA scaler
with open('wnba_scaler.pkl', 'rb') as f:
    scaler = pickle.load(f)
print(f"Scaler loaded: {scaler.n_features_in_} features")

# Define features and targets — now includes 5 H2H features (57 total)
target_cols = ['PTS', 'REB', 'AST', 'BLK', 'STL', 'FG3M']

feature_cols = [
    'PTS_roll5', 'REB_roll5', 'AST_roll5', 'BLK_roll5',
    'STL_roll5', 'FG3M_roll5', 'MIN_roll5', 'TOV_roll5',
    'FGA_roll5', 'FG3A_roll5',
    'PTS_roll10', 'REB_roll10', 'AST_roll10', 'BLK_roll10',
    'STL_roll10', 'FG3M_roll10', 'MIN_roll10', 'TOV_roll10',
    'FGA_roll10', 'FG3A_roll10',
    'HOME_AWAY', 'DAYS_REST', 'DEF_RATING', 'PACE',
    'OPP_PTS_ALLOWED_PG', 'OPP_REB_ALLOWED_PG', 'OPP_AST_ALLOWED_PG',
    'OPP_BLK_PG', 'OPP_STL_PG', 'OPP_3PM_ALLOWED_PG',
    'USG_PCT',
    'OPP_PTS_VS_POS', 'OPP_REB_VS_POS', 'OPP_AST_VS_POS',
    'OPP_BLK_VS_POS', 'OPP_STL_VS_POS', 'OPP_3PM_VS_POS',
    'USG_PCT_roll5',
    'OPP_PTS_VS_POS_roll5_norm', 'OPP_REB_VS_POS_roll5_norm',
    'OPP_AST_VS_POS_roll5_norm', 'OPP_BLK_VS_POS_roll5_norm',
    'OPP_STL_VS_POS_roll5_norm', 'OPP_3PM_VS_POS_roll5_norm',
    'RELATIVE_USG', 'USG_RANK',
    'PTS_std_roll10', 'REB_std_roll10',
    'PTS_cv_roll10', 'REB_cv_roll10',
    'IS_ROOKIE_SEASON',
    'GAMES_PLAYED',
    'H2H_PTS_AVG', 'H2H_REB_AVG', 'H2H_AST_AVG',
    'H2H_FG3M_AVG', 'H2H_GAMES',
]
print(f"Features: {len(feature_cols)}")

# Define model
class PlayerPropModel(nn.Module):
    def __init__(self, input_dim, target_stats):
        super().__init__()
        self.trunk = nn.Sequential(
            nn.Linear(input_dim, 128), nn.ReLU(),
            nn.BatchNorm1d(128), nn.Dropout(0.5),
            nn.Linear(128, 64), nn.ReLU(),
            nn.BatchNorm1d(64), nn.Dropout(0.4),
        )
        self.heads = nn.ModuleDict({
            stat: nn.Sequential(
                nn.Linear(64, 32), nn.ReLU(), nn.Linear(32, 2)
            ) for stat in target_stats
        })
    def forward(self, x):
        shared = self.trunk(x)
        outputs = {}
        for stat, head in self.heads.items():
            raw       = head(shared)
            mu        = raw[:, 0]
            log_sigma = torch.clamp(raw[:, 1], min=-3, max=3)
            sigma     = torch.exp(log_sigma) + 1e-6
            outputs[stat] = (mu, sigma)
        return outputs

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model  = PlayerPropModel(input_dim=len(feature_cols), target_stats=target_cols)
model.load_state_dict(torch.load('wnba_best_model.pth', map_location=device))
model.eval()
print(f"WNBA model loaded ✅")
print(f"Device: {device}")

# Build team and position lookup tables for TONIGHT'S opponent context
team_stat_cols = ['DEF_RATING', 'PACE', 'OPP_PTS_ALLOWED_PG', 'OPP_REB_ALLOWED_PG',
                   'OPP_AST_ALLOWED_PG', 'OPP_BLK_PG', 'OPP_STL_PG', 'OPP_3PM_ALLOWED_PG']

team_lookup = (
    df_master[df_master['SEASON'] == CURRENT_SEASON]
    .drop_duplicates(subset=['OPPONENT'])
    .set_index('OPPONENT')[team_stat_cols]
)

pos_stat_cols = ['OPP_PTS_VS_POS', 'OPP_REB_VS_POS', 'OPP_AST_VS_POS',
                  'OPP_BLK_VS_POS', 'OPP_STL_VS_POS', 'OPP_3PM_VS_POS',
                  'OPP_PTS_VS_POS_roll5_norm', 'OPP_REB_VS_POS_roll5_norm',
                  'OPP_AST_VS_POS_roll5_norm', 'OPP_BLK_VS_POS_roll5_norm',
                  'OPP_STL_VS_POS_roll5_norm', 'OPP_3PM_VS_POS_roll5_norm']

pos_lookup = (
    df_master[df_master['SEASON'] == CURRENT_SEASON]
    .drop_duplicates(subset=['OPPONENT', 'POSITION'])
    .set_index(['OPPONENT', 'POSITION'])[pos_stat_cols]
)

print(f"✅ Team lookup built — {len(team_lookup)} teams")
print(f"✅ Position lookup built — {len(pos_lookup)} opponent/position pairs")


def get_h2h_features(player_name, opponent_abbr, df_master,
                      h2h_stats=['PTS', 'REB', 'AST', 'FG3M']):
    """Fresh H2H averages for player vs a SPECIFIC opponent (last 5 meetings)."""
    history = df_master[
        (df_master['PLAYER_NAME'] == player_name) &
        (df_master['OPPONENT'] == opponent_abbr)
    ].sort_values('GAME_DATE')

    games_count = len(history)
    if games_count == 0:
        return None, 0

    recent = history.tail(5)
    h2h = {f'H2H_{stat}_AVG': recent[stat].mean() for stat in h2h_stats}
    return h2h, games_count

print(f"✅ get_h2h_features defined")

Dataset loaded: 15,882 rows
Scaler loaded: 57 features
Features: 57
WNBA model loaded ✅
Device: cpu
✅ Team lookup built — 15 teams
✅ Position lookup built — 45 opponent/position pairs
✅ get_h2h_features defined


In [5]:
# Prediction function — opponent-aware
# Run once per session
def predict_player(player_name, opponent_abbr, df_master, model, scaler,
                    feature_cols, target_cols):
    """
    Player-form features come from their most recent complete game.
    Opponent-context features (DEF_RATING, PACE, opponent allowed,
    opponent-vs-position, H2H) are pulled fresh for TONIGHT's actual
    opponent_abbr — not whoever the player last played.
    """
    player_df = df_master[df_master['PLAYER_NAME'] == player_name].copy()
    if len(player_df) == 0:
        return None

    player_df = player_df.sort_values('GAME_DATE')

    if player_df[feature_cols].iloc[-1].isna().any():
        complete_rows = player_df.dropna(subset=feature_cols)
        if len(complete_rows) == 0:
            return None
        latest = complete_rows.iloc[-1].copy()
    else:
        latest = player_df.iloc[-1].copy()

    game_count = len(player_df)
    position   = latest['POSITION']

    if opponent_abbr in team_lookup.index:
        for col in team_stat_cols:
            latest[col] = team_lookup.loc[opponent_abbr, col]
    else:
        latest[team_stat_cols] = df_master[
            df_master['SEASON'] == CURRENT_SEASON
        ][team_stat_cols].mean()

    pos_key = (opponent_abbr, position)
    if pos_key in pos_lookup.index:
        for col in pos_stat_cols:
            latest[col] = pos_lookup.loc[pos_key, col]
    else:
        latest[pos_stat_cols] = df_master[
            df_master['SEASON'] == CURRENT_SEASON
        ][pos_stat_cols].mean()

    h2h, h2h_games = get_h2h_features(player_name, opponent_abbr, df_master)
    if h2h is not None:
        for col, val in h2h.items():
            latest[col] = val
        latest['H2H_GAMES'] = h2h_games
    else:
        for stat in ['PTS', 'REB', 'AST', 'FG3M']:
            latest[f'H2H_{stat}_AVG'] = latest[f'{stat}_roll5']
        latest['H2H_GAMES'] = 0

    features = latest[feature_cols].values.astype(np.float32).reshape(1, -1)
    features = scaler.transform(features)

    if np.isnan(features).any():
        return None

    feature_tensor = torch.tensor(features, dtype=torch.float32).to(device)

    with torch.no_grad():
        outputs = model(feature_tensor)

    results = {}
    for stat in target_cols:
        mu    = outputs[stat][0].item()
        sigma = outputs[stat][1].item()
        results[stat] = {'mu': mu, 'sigma': sigma}

    return results, latest['GAME_DATE'], game_count

In [7]:
# Pull props for today and convert times
# Re-run freely to refresh props (API calls limited to 500 per day)

from datetime import datetime, timezone, timedelta

# EST timezone
EST = timezone(timedelta(hours=-4))

# Pull tonight's WNBA props from DraftKings
url      = 'https://api.the-odds-api.com/v4/sports/basketball_wnba/odds'
response = requests.get(url, params={
    'apiKey':    ODDS_API_KEY,
    'regions':   'us',
    'oddsFormat': 'american',
})
games = response.json()

print(f"WNBA games available: {len(games)}")
for g in games:
    utc_time = datetime.fromisoformat(g['commence_time'].replace('Z', '+00:00'))
    est_time = utc_time.astimezone(EST)
    print(f"  {g['away_team']} @ {g['home_team']}  —  {est_time.strftime('%I:%M %p EST  %Y-%m-%d')}")
print()

# Filter to only today's games in EST
today_est   = datetime.now(EST).strftime('%Y-%m-%d')
games_today = []

for g in games:
    utc_time = datetime.fromisoformat(g['commence_time'].replace('Z', '+00:00'))
    est_time = utc_time.astimezone(EST)
    if est_time.strftime('%Y-%m-%d') == today_est:
        games_today.append(g)

print(f"Games today in EST ({today_est}): {len(games_today)}")
for g in games_today:
    utc_time = datetime.fromisoformat(g['commence_time'].replace('Z', '+00:00'))
    est_time = utc_time.astimezone(EST)
    print(f"  {g['away_team']} @ {g['home_team']}  —  {est_time.strftime('%I:%M %p EST')}")
print()

# Use today's games for props pull
games = games_today

# Step 2 — Pull props for each game
MARKETS = [
    'player_points',
    'player_rebounds',
    'player_assists',
    'player_threes',
]

all_props = []

for game in games:
    game_id   = game['id']
    home_team = game['home_team']
    away_team = game['away_team']

    for market in MARKETS:
        url      = f'https://api.the-odds-api.com/v4/sports/basketball_wnba/events/{game_id}/odds'
        response = requests.get(url, params={
            'apiKey':     ODDS_API_KEY,
            'regions':    'us',
            'markets':    market,
            'bookmakers': 'draftkings',
            'oddsFormat': 'american',
        })
        time.sleep(0.5)

        if response.status_code != 200:
            print(f"❌ Failed {market} for {away_team} @ {home_team}")
            continue

        data = response.json()

        for bookmaker in data.get('bookmakers', []):
            for mkt in bookmaker.get('markets', []):
                players_seen = {}
                for outcome in mkt.get('outcomes', []):
                    player = outcome['description']
                    side   = outcome['name']
                    price  = outcome['price']
                    line   = outcome['point']

                    if player not in players_seen:
                        players_seen[player] = {
                            'player':    player,
                            'stat':      market,
                            'line':      line,
                            'home_team': home_team,
                            'away_team': away_team,
                        }

                    if side == 'Over':
                        players_seen[player]['over_juice']  = price
                    else:
                        players_seen[player]['under_juice'] = price

                all_props.extend(players_seen.values())

market_to_stat = {
    'player_points':   'PTS',
    'player_rebounds': 'REB',
    'player_assists':  'AST',
    'player_threes':   'FG3M',
}

df_props = pd.DataFrame(all_props)
df_props['stat'] = df_props['stat'].map(market_to_stat)

print(f"Total props pulled: {len(df_props)}")
print(f"Requests remaining: {response.headers.get('x-requests-remaining')}")
print()
print(df_props[['player', 'stat', 'line',
                'over_juice', 'under_juice']].head(10))

WNBA games available: 5
  Atlanta Dream @ Connecticut Sun  —  07:00 PM EST  2026-08-13
  Los Angeles Sparks @ New York Liberty  —  08:00 PM EST  2026-08-13
  Washington Mystics @ Las Vegas Aces  —  10:00 PM EST  2026-08-13
  Dallas Wings @ Indiana Fever  —  07:30 PM EST  2026-08-14
  Portland Fire @ Seattle Storm  —  10:00 PM EST  2026-08-14

Games today in EST (2026-08-13): 3
  Atlanta Dream @ Connecticut Sun  —  07:00 PM EST
  Los Angeles Sparks @ New York Liberty  —  08:00 PM EST
  Washington Mystics @ Las Vegas Aces  —  10:00 PM EST

Total props pulled: 92
Requests remaining: 474

            player stat  line  over_juice  under_juice
0     Allisha Gray  PTS  18.5        -124         -107
1     Rhyne Howard  PTS  16.5        -124         -107
2      Angel Reese  PTS  16.5         100         -132
3  Brittney Griner  PTS  12.5        -111         -119
4      Leïla Lacan  PTS  12.5         100         -132
5    Jordin Canada  PTS  11.5        -107         -124
6   Diamond Miller  PTS

In [8]:
# Pull injury report
# Re-run freely to get fresh injury reports

def get_injury_report():
    """Pulls WNBA injury report from ESPN API."""
    try:
        url      = 'https://site.api.espn.com/apis/site/v2/sports/basketball/wnba/injuries'
        response = requests.get(url, timeout=10)

        if response.status_code != 200:
            print(f"❌ Injury report failed: {response.status_code}")
            return {}

        data     = response.json()
        injuries = {}

        for team in data.get('injuries', []):
            team_name = team.get('team', {}).get('displayName', '')
            for injury in team.get('injuries', []):
                player_name = injury.get('athlete', {}).get('displayName', '')
                status      = injury.get('status', '')
                details     = injury.get('shortComment', '')
                injuries[player_name] = {
                    'status':  status,
                    'details': details,
                    'team':    team_name,
                }

        return injuries

    except Exception as e:
        print(f"Error: {e}")
        return {}

injury_report = get_injury_report()

print(f"Players on injury report: {len(injury_report)}")
print()

# Check tonight's players
tonight_players = df_props['player'].unique()
flagged = []
for player in sorted(tonight_players):
    if player in injury_report:
        info = injury_report[player]
        print(f"  {player:<28} {info['status']:<15} {info['details'][:50]}")
        flagged.append(player)

if len(flagged) == 0:
    print("  No players from tonight's props on injury report ✅")
print()
print(f"Total flagged: {len(flagged)}")

Players on injury report: 47

  Brittney Griner              Day-To-Day      Griner (knee) is questionable for Thursday's game 

Total flagged: 1


In [9]:
# Run predictions and print ranked table
# Re-run freely to get fresh predictions on props

# Calibration offsets — based on retrained model bias check (June 21 refresh)
CALIBRATION = {
    'PTS':  +0.4,
    'REB':  +0.2,
    'AST':  +0.3,
    'FG3M': +0.1,
}

# Edge thresholds by confidence tier
MIN_EDGE_GREEN  = 20.0   # 80+ games  — trust smaller edges
MIN_EDGE_YELLOW = 20.0   # 40-79 games — need more conviction
MIN_EDGE_RED    = 25.0   # <40 games  — only very high edges

# Stat-specific filters based on performance analysis
# REB/FG3M skipped from BETTING — still tracked via Cell 7 for data collection
# AST: model genuinely predicting well (75% on high-edge)
# PTS: improving with edge
SKIP_STATS = ['REB', 'FG3M']


def american_to_prob(juice):
    if juice < 0:
        return abs(juice) / (abs(juice) + 100)
    else:
        return 100 / (juice + 100)

def confidence_tier(game_count):
    if game_count >= 80:
        return '🟢'
    elif game_count >= 40:
        return '🟡'
    else:
        return '🔴'

def get_injury_flag(player_name, injury_report):
    if player_name in injury_report:
        status = injury_report[player_name]['status'].lower()
        if status == 'out':
            return 'OUT'
        elif status == 'doubtful':
            return 'DOUBT'
        elif status == 'questionable':
            return 'QUEST'
        else:
            return 'INACT'
    return ''

# Run model on every prop
results_list = []

for _, row in df_props.iterrows():
    player = row['player']
    stat   = row['stat']
    line   = row['line']

    if 'over_juice' not in row or 'under_juice' not in row:
        continue
    if pd.isna(row.get('over_juice')) or pd.isna(row.get('under_juice')):
        continue

    # Determine player's team and tonight's actual opponent
    player_history = df_master[df_master['PLAYER_NAME'] == player]
    if len(player_history) == 0:
        continue

    player_team_abbr = player_history.sort_values('GAME_DATE')['PLAYER_TEAM'].iloc[-1]
    home_abbr = reverse_team_map.get(row['home_team'])
    away_abbr = reverse_team_map.get(row['away_team'])

    if player_team_abbr == home_abbr:
        opponent_abbr = away_abbr
    elif player_team_abbr == away_abbr:
        opponent_abbr = home_abbr
    else:
        # Player's team doesn't match tonight's game — likely traded/inactive
        continue

    pred = predict_player(
        player, opponent_abbr, df_master, model, scaler, feature_cols, target_cols
    )

    if pred is None:
        continue

    predictions, last_game, game_count = pred

    if stat not in predictions:
        continue

    mu    = predictions[stat]['mu']
    sigma = predictions[stat]['sigma']

    # Apply calibration offset
    mu_calibrated = mu + CALIBRATION.get(stat, 0)

    prob_over  = 1 - scipy_norm.cdf(line, mu_calibrated, sigma)
    prob_under = scipy_norm.cdf(line, mu_calibrated, sigma)

    breakeven_over  = american_to_prob(row['over_juice'])
    breakeven_under = american_to_prob(row['under_juice'])

    edge_over  = prob_over  - breakeven_over
    edge_under = prob_under - breakeven_under

    if edge_over > edge_under and edge_over > 0:
        recommendation = 'OVER'
        edge = edge_over
    elif edge_under > edge_over and edge_under > 0:
        recommendation = 'UNDER'
        edge = edge_under
    else:
        recommendation = 'NO BET'
        edge = max(edge_over, edge_under)

    injury_flag = get_injury_flag(player, injury_report)

    results_list.append({
        'player':         player,
        'stat':           stat,
        'line':           line,
        'mu':             round(mu_calibrated, 1),
        'mu_raw':         round(mu, 1),
        'sigma':          round(sigma, 1),
        'recommendation': recommendation,
        'edge':           round(edge * 100, 1),
        'over_juice':     row['over_juice'],
        'under_juice':    row['under_juice'],
        'game':           f"{row['away_team']} @ {row['home_team']}",
        'opponent':       opponent_abbr,
        'data_through':   str(last_game.date()),
        'game_count':     game_count,
        'confidence':     confidence_tier(game_count),
        'injury':         injury_flag,
    })

df_results = pd.DataFrame(results_list)

# Separate injured players
df_injured = df_results[df_results['injury'] == 'OUT']

# Filter bets with tiered edge thresholds and stat-specific rules
df_bets = df_results[
    (df_results['recommendation'] != 'NO BET') &
    (~df_results['injury'].isin(['OUT', 'INACT'])) &
    (~df_results['stat'].isin(SKIP_STATS)) &
    (df_results['stat'].isin(['PTS', 'AST'])) &
    (
        ((df_results['game_count'] >= 80) &
         (df_results['edge'] >= MIN_EDGE_GREEN)) |
        ((df_results['game_count'] >= 40) &
         (df_results['game_count'] < 80) &
         (df_results['edge'] >= MIN_EDGE_YELLOW)) |
        ((df_results['game_count'] < 40) &
         (df_results['edge'] >= MIN_EDGE_RED))
    )
].sort_values('edge', ascending=False).reset_index(drop=True)

# Print ranked table
print(f"{'='*105}")
print(f"  TONIGHT'S WNBA BEST BETS — DraftKings Props  "
      f"(⚙️  Calibration: PTS+1.3 REB+0.2 AST+0.3 FG3M+0.1 | "
      f"REB/FG3M skipped from betting)")
print(f"{'='*105}")
print(f"{'#':<4} {'C':<3} {'Player':<28} {'Stat':<6} {'Line':<7} "
      f"{'Pred μ':<8} {'Rec':<7} {'Edge':>6}  {'Juice':<8} {'Games'}")
print(f"{'-'*105}")

for i, row in df_bets.iterrows():
    juice     = row['over_juice'] if row['recommendation'] == 'OVER' else row['under_juice']
    juice_str = f"+{juice}" if juice > 0 else str(juice)
    inj       = f" ⚠️ {row['injury']}" if row['injury'] else ''
    print(
        f"{i+1:<4} {row['confidence']:<3} {row['player']:<28} {row['stat']:<6} "
        f"{row['line']:<7} {row['mu']:<8} {row['recommendation']:<7} "
        f"{row['edge']:>5.1f}%  {juice_str:<8} {row['game_count']}"
        f"{inj}"
    )

print(f"{'='*105}")
print()

if len(df_injured) > 0:
    print(f"🚫 REMOVED — Player listed as OUT or INACT:")
    for _, row in df_injured.iterrows():
        if row['recommendation'] != 'NO BET':
            print(f"   {row['player']:<28} {row['stat']:<6} "
                  f"Line {row['line']}  Edge {row['edge']:.1f}%  ← removed")
    print()

pts_bets = len(df_bets[df_bets['stat'] == 'PTS'])
ast_bets = len(df_bets[df_bets['stat'] == 'AST'])

high   = len(df_bets[df_bets['game_count'] >= 80])
medium = len(df_bets[(df_bets['game_count'] >= 40) & (df_bets['game_count'] < 80)])
low    = len(df_bets[df_bets['game_count'] < 40])

print(f"🟢 High confidence (80+ games):    {high} bets  (min edge {MIN_EDGE_GREEN}%)")
print(f"🟡 Medium confidence (40-79 games): {medium} bets  (min edge {MIN_EDGE_YELLOW}%)")
print(f"🔴 Low confidence (<40 games):      {low} bets  (min edge {MIN_EDGE_RED}%)")
print()
print(f"By stat:  PTS {pts_bets}  AST {ast_bets}  REB/FG3M skipped from betting")
print()
print(f"Total +EV bets shown: {len(df_bets)}")
print(f"Total props scanned:  {len(df_results)}")
print(f"Removed (OUT/INACT):  {len(df_injured[df_injured['recommendation'] != 'NO BET'])}")
print()
print(f"⚙️  Model retrained with H2H features (val loss 1.8788)")
print(f"⚙️  Opponent context now matched to tonight's actual matchup")

  TONIGHT'S WNBA BEST BETS — DraftKings Props  (⚙️  Calibration: PTS+1.3 REB+0.2 AST+0.3 FG3M+0.1 | REB/FG3M skipped from betting)
#    C   Player                       Stat   Line    Pred μ   Rec       Edge  Juice    Games
---------------------------------------------------------------------------------------------------------
1    🟢   Erica Wheeler                AST    5.5     3.6      UNDER    28.7%  -116     166
2    🟢   Sabrina Ionescu              PTS    18.5    13.1     UNDER    28.5%  -112     166
3    🟢   Shakira Austin               PTS    17.5    12.7     UNDER    27.4%  -124     99
4    🟢   Nneka Ogwumike               AST    3.5     2.1      UNDER    25.4%  -147     166
5    🟡   Rae Burrell                  PTS    15.5    11.8     UNDER    22.1%  -107     65
6    🟢   Sabrina Ionescu              AST    5.5     3.9      UNDER    21.4%  -131     166

🟢 High confidence (80+ games):    5 bets  (min edge 20.0%)
🟡 Medium confidence (40-79 games): 1 bets  (min edge 20.0%)
🔴 Low 

In [134]:
# Bias check history tracker
# Logs each night's systematic bias check so we can see the trend
# over time instead of reacting to a single night's sample

import json
import os

BIAS_LOG_FILE = 'wnba_bias_log.json'

def load_bias_log():
    if os.path.exists(BIAS_LOG_FILE):
        with open(BIAS_LOG_FILE, 'r') as f:
            return json.load(f)
    return []

def save_bias_log(log):
    with open(BIAS_LOG_FILE, 'w') as f:
        json.dump(log, f, indent=2)

def log_tonights_bias_check(df_results, calibration, today=None):
    """
    Computes and saves tonight's per-stat bias check.
    Safe to re-run same night — overwrites today's entry instead of duplicating.
    """
    today = today or datetime.now().strftime('%Y-%m-%d')
    log   = load_bias_log()

    # Remove any existing entry for today (so re-running tonight doesn't duplicate)
    log = [entry for entry in log if entry['date'] != today]

    for stat in ['PTS', 'REB', 'AST', 'FG3M']:
        stat_rows = df_results[df_results['stat'] == stat]
        if len(stat_rows) == 0:
            continue

        avg_raw_mu = stat_rows['mu_raw'].mean()
        avg_line   = stat_rows['line'].mean()
        gap        = avg_line - avg_raw_mu
        offset     = calibration.get(stat, 0)
        net_gap    = gap - offset

        log.append({
            'date':           today,
            'stat':           stat,
            'sample_size':    len(stat_rows),
            'avg_raw_mu':     round(avg_raw_mu, 2),
            'avg_line':       round(avg_line, 2),
            'gap':            round(gap, 2),
            'offset':         offset,
            'remaining_gap':  round(net_gap, 2),
        })

    save_bias_log(log)
    print(f"✅ Logged bias check for {today}")
    return log


def print_bias_trend(n_nights=10):
    """
    Prints the remaining_gap trend per stat across the last n_nights.
    """
    log = load_bias_log()
    if len(log) == 0:
        print("No bias history logged yet.")
        return

    dates = sorted(set(entry['date'] for entry in log), reverse=True)[:n_nights]
    dates = sorted(dates)  # chronological for display

    print(f"{'='*70}")
    print(f"  BIAS CHECK TREND — last {len(dates)} nights")
    print(f"{'='*70}")

    for stat in ['PTS', 'REB', 'AST', 'FG3M']:
        stat_entries = [e for e in log if e['stat'] == stat and e['date'] in dates]
        if len(stat_entries) == 0:
            continue

        stat_entries = sorted(stat_entries, key=lambda x: x['date'])

        print(f"\n  {stat}")
        print(f"  {'Date':<12} {'Sample':>7} {'Raw μ':>7} {'DK Line':>8} "
              f"{'Gap':>6} {'Offset':>7} {'Remaining':>10}")
        for e in stat_entries:
            print(f"  {e['date']:<12} {e['sample_size']:>7} {e['avg_raw_mu']:>7.1f} "
                  f"{e['avg_line']:>8.1f} {e['gap']:>+6.1f} {e['offset']:>+7.1f} "
                  f"{e['remaining_gap']:>+10.1f}")

        avg_remaining = sum(e['remaining_gap'] for e in stat_entries) / len(stat_entries)
        print(f"  {'→ Average remaining gap:':<45} {avg_remaining:>+.2f}")

    print(f"\n{'='*70}")


# Run tonight's log
from datetime import datetime
log_tonights_bias_check(df_results, CALIBRATION)
print()
print_bias_trend()

✅ Logged bias check for 2026-06-30

  BIAS CHECK TREND — last 4 nights

  PTS
  Date          Sample   Raw μ  DK Line    Gap  Offset  Remaining
  2026-06-24        38    12.8     13.0   +0.2    +1.3       -1.1
  2026-06-25        31    12.1     13.2   +1.2    +1.3       -0.1
  2026-06-26        27    12.6     12.4   -0.2    +1.3       -1.5
  2026-06-30        10    12.3     13.2   +0.8    +0.4       +0.5
  → Average remaining gap:                      -0.57

  REB
  Date          Sample   Raw μ  DK Line    Gap  Offset  Remaining
  2026-06-24        34     4.9      5.0   +0.1    +0.2       -0.1
  2026-06-25        27     4.7      5.2   +0.5    +0.2       +0.3
  2026-06-26        22     5.2      5.3   +0.0    +0.2       -0.2
  2026-06-30         8     4.8      5.4   +0.6    +0.2       +0.4
  → Average remaining gap:                      +0.11

  AST
  Date          Sample   Raw μ  DK Line    Gap  Offset  Remaining
  2026-06-24        18     4.2      4.4   +0.2    +0.3       -0.1
  2026-0

In [102]:
# ── TEMPORARY OVERRIDE — June 24 ──────────────────────────
# Holding off on real bets tonight while we validate the new
# opponent-fix + retrain. Track everything, place nothing.
# Remove this override once we're confident in calibration again.

#PAUSE_LIVE_BETTING = True

#if PAUSE_LIVE_BETTING:
#    df_bets_for_logging = df_bets.iloc[0:0]  # empty — nothing flagged as placed
#    print("⏸️  LIVE BETTING PAUSED — tonight's bets will log as TRACKING ONLY")
#else:
#    df_bets_for_logging = df_bets

⏸️  LIVE BETTING PAUSED — tonight's bets will log as TRACKING ONLY


In [136]:
# CLV tracking
# Logs ALL 5%+ edge bets for data collection
# Betting recommendations still use 20%+ filter from Cell 6

import json
import os
from datetime import datetime

WNBA_BETS_LOG_FILE = 'wnba_bets_log.json'

def load_bets_log():
    if os.path.exists(WNBA_BETS_LOG_FILE):
        with open(WNBA_BETS_LOG_FILE, 'r') as f:
            return json.load(f)
    return []

def save_bets_log(log):
    import numpy as np
    def convert(obj):
        if isinstance(obj, (np.integer, np.int64)):
            return int(obj)
        elif isinstance(obj, (np.floating, np.float64)):
            return float(obj)
        elif isinstance(obj, np.ndarray):
            return obj.tolist()
        return obj
    clean_log = [{k: convert(v) for k, v in bet.items()} for bet in log]
    with open(WNBA_BETS_LOG_FILE, 'w') as f:
        json.dump(clean_log, f, indent=2)


def log_todays_bets(df_tracking, injury_report, min_edge=5.0):
    """
    Logs all 5%+ edge bets for data collection.
    Uses placed_keys to flag which bets were actually placed (20%+).
    """
    log        = load_bets_log()
    today      = datetime.now().strftime('%Y-%m-%d')
    bets_added = 0

    # Build set of actual placed bets from df_bets (20%+ filter)
    placed_keys = set(zip(df_bets_for_logging['player'], df_bets_for_logging['stat']))

    for _, row in df_tracking[df_tracking['edge'] >= min_edge].iterrows():
        juice = row['over_juice'] if row['recommendation'] == 'OVER' else row['under_juice']

        bet_entry = {
            'date':           today,
            'player':         row['player'],
            'stat':           row['stat'],
            'line':           row['line'],
            'recommendation': row['recommendation'],
            'edge':           row['edge'],
            'juice':          juice,
            'pred_mu':        row['mu'],
            'pred_sigma':     row['sigma'],
            'game':           row['game'],
            'game_count':     row['game_count'],
            'confidence':     row['confidence'],
            'data_through':   row['data_through'],
            'closing_line':   None,
            'actual_result':  None,
            'bet_result':     None,
            'clv':            None,
            'bet_placed':     (row['player'], row['stat']) in placed_keys,
        }

        is_duplicate = any(
            b['date']   == today and
            b['player'] == row['player'] and
            b['stat']   == row['stat']
            for b in log
        )

        if not is_duplicate:
            log.append(bet_entry)
            bets_added += 1

    save_bets_log(log)
    print(f"✅ Logged {bets_added} WNBA bets for {today}")
    return log


def print_performance_summary():
    log = load_bets_log()
    if len(log) == 0:
        print("No bets logged yet.")
        return

    completed = [b for b in log if b['bet_result'] is not None
                 and b['bet_result'] != 'VOID']
    pending   = [b for b in log if b['bet_result'] is None]
    voided    = [b for b in log if b['bet_result'] == 'VOID']

    # Split into placed vs tracking
    placed   = [b for b in completed if b.get('bet_placed', True)]
    tracking = [b for b in completed if not b.get('bet_placed', True)]

    print(f"{'='*60}")
    print(f"  WNBA PERFORMANCE SUMMARY")
    print(f"{'='*60}")
    print(f"  Total bets logged:   {len(log)}")
    print(f"  Completed:           {len(completed)}")
    print(f"  Pending:             {len(pending)}")
    print(f"  Voided:              {len(voided)}")
    print()

    if len(completed) == 0:
        print("  No completed bets yet.")
        return

    # Placed bets performance
    if len(placed) > 0:
        wins     = len([b for b in placed if b['bet_result'] == 'WIN'])
        losses   = len([b for b in placed if b['bet_result'] == 'LOSS'])
        pushes   = len([b for b in placed if b['bet_result'] == 'PUSH'])
        win_rate = wins / (wins + losses) if (wins + losses) > 0 else 0

        print(f"  ── PLACED BETS (20%+ edge) ──────────────────")
        print(f"  Record:              {wins}W - {losses}L - {pushes}P")
        print(f"  Win rate:            {win_rate:.1%}")
        print()

        print(f"  By stat (placed only):")
        for stat in ['PTS', 'REB', 'AST', 'FG3M']:
            stat_bets = [b for b in placed if b['stat'] == stat]
            if len(stat_bets) == 0:
                continue
            stat_wins = len([b for b in stat_bets if b['bet_result'] == 'WIN'])
            stat_loss = len(stat_bets) - stat_wins
            print(f"    {stat:<6} {stat_wins}W/{stat_loss}L  "
                  f"({stat_wins/len(stat_bets):.0%})  [{len(stat_bets)} bets]")

        print()
        print(f"  By confidence (placed only):")
        for tier in ['🟢', '🟡', '🔴']:
            tier_bets = [b for b in placed if b['confidence'] == tier]
            if len(tier_bets) == 0:
                continue
            tier_wins = len([b for b in tier_bets if b['bet_result'] == 'WIN'])
            tier_loss = len(tier_bets) - tier_wins
            print(f"    {tier}  {tier_wins}W/{tier_loss}L  "
                  f"({tier_wins/len(tier_bets):.0%})  [{len(tier_bets)} bets]")

        print()

    # Tracking only performance
    if len(tracking) > 0:
        t_wins     = len([b for b in tracking if b['bet_result'] == 'WIN'])
        t_losses   = len([b for b in tracking if b['bet_result'] == 'LOSS'])
        t_win_rate = t_wins / (t_wins + t_losses) if (t_wins + t_losses) > 0 else 0

        print(f"  ── TRACKING ONLY (5-20% edge) ───────────────")
        print(f"  Record:              {t_wins}W - {t_losses}L")
        print(f"  Win rate:            {t_win_rate:.1%}")
        print()

        print(f"  By stat (tracking only):")
        for stat in ['PTS', 'REB', 'AST', 'FG3M']:
            stat_bets = [b for b in tracking if b['stat'] == stat]
            if len(stat_bets) == 0:
                continue
            stat_wins = len([b for b in stat_bets if b['bet_result'] == 'WIN'])
            stat_loss = len(stat_bets) - stat_wins
            print(f"    {stat:<6} {stat_wins}W/{stat_loss}L  "
                  f"({stat_wins/len(stat_bets):.0%})  [{len(stat_bets)} bets]")

    print(f"{'='*60}")


# ── Build tracking dataframe — all 5%+ bets ──────────────
df_tracking = df_results[
    (df_results['recommendation'] != 'NO BET') &
    (~df_results['injury'].isin(['OUT', 'INACT'])) &
    (df_results['edge'] >= 5.0)
].reset_index(drop=True)

# Log all tracking bets
log = log_todays_bets(df_tracking, injury_report, min_edge=5.0)
print()

# Show what was logged — split by placed vs tracking
today          = datetime.now().strftime('%Y-%m-%d')
todays_bets    = [b for b in log if b['date'] == today]
placed_today   = [b for b in todays_bets if b.get('bet_placed', True)]
tracking_today = [b for b in todays_bets if not b.get('bet_placed', True)]

print(f"✅ PLACED BETS tonight ({len(placed_today)}) — these are your actual bets:")
print(f"{'Player':<28} {'Stat':<6} {'Line':<7} {'Rec':<7} {'Edge':>6}")
print("-" * 60)
for bet in placed_today:
    print(f"{bet['player']:<28} {bet['stat']:<6} {bet['line']:<7} "
          f"{bet['recommendation']:<7} {bet['edge']:>5.1f}%")

print()
print(f"📊 TRACKING ONLY tonight ({len(tracking_today)}) — logged for data, not bet:")
print(f"{'Player':<28} {'Stat':<6} {'Line':<7} {'Rec':<7} {'Edge':>6}")
print("-" * 60)
for bet in tracking_today:
    print(f"{bet['player']:<28} {bet['stat']:<6} {bet['line']:<7} "
          f"{bet['recommendation']:<7} {bet['edge']:>5.1f}%")

✅ Logged 13 WNBA bets for 2026-06-30

✅ PLACED BETS tonight (0) — these are your actual bets:
Player                       Stat   Line    Rec       Edge
------------------------------------------------------------

📊 TRACKING ONLY tonight (13) — logged for data, not bet:
Player                       Stat   Line    Rec       Edge
------------------------------------------------------------
Jackie Young                 PTS    19.5    UNDER     8.6%
Jonquel Jones                PTS    15.5    UNDER     8.4%
Chennedy Carter              PTS    13.5    OVER     12.2%
Sabrina Ionescu              PTS    13.5    UNDER     8.7%
NaLyssa Smith                PTS    12.5    UNDER    16.3%
Marine Johannes              PTS    6.5     OVER     20.5%
Jackie Young                 REB    5.5     UNDER    14.1%
Sabrina Ionescu              REB    4.5     UNDER    11.4%
Pauline Astier               REB    2.5     OVER      6.0%
Leonie Fiebich               REB    2.5     OVER      8.5%
Sabrina Ionescu   

In [187]:
#import json

# Remove today's bets from the log and re-log with filtered list
#log       = load_bets_log()
#today     = '2026-06-06'

# Remove all of today's pending bets
#log_clean = [b for b in log if b['date'] != today]

#save_bets_log(log_clean)
#print(f"Removed today's bets from log")
#print(f"Remaining bets in log: {len(log_clean)}")

In [9]:
# Automated results updater — function definitions
# Run once per session

from nba_api.stats.endpoints import playergamelog
import time
import numpy as np

def save_closing_lines(df_bets, min_edge=5.0, game_filter=None):
    """
    Saves closing lines for logged bets.
    Use game_filter to save lines for specific tipoff windows.

    Example:
      save_closing_lines(df_bets, game_filter='Las Vegas Aces')
      save_closing_lines(df_bets)  # saves all games at once
    """
    log   = load_bets_log()
    today = datetime.now().strftime('%Y-%m-%d')

    url      = 'https://api.the-odds-api.com/v4/sports/basketball_wnba/odds'
    response = requests.get(url, params={
        'apiKey':    ODDS_API_KEY,
        'regions':   'us',
        'oddsFormat': 'american',
    })
    games = response.json()

    if game_filter:
        games = [
            g for g in games
            if game_filter in g['home_team'] or game_filter in g['away_team']
        ]
        print(f"Filtering to games involving: {game_filter}")
        print(f"Games found: {len(games)}")

    current_lines = {}

    for game in games:
        game_id = game['id']
        for market in ['player_points', 'player_rebounds',
                       'player_assists', 'player_threes']:

            url      = f'https://api.the-odds-api.com/v4/sports/basketball_wnba/events/{game_id}/odds'
            response = requests.get(url, params={
                'apiKey':     ODDS_API_KEY,
                'regions':    'us',
                'markets':    market,
                'bookmakers': 'draftkings',
                'oddsFormat': 'american',
            })
            time.sleep(0.5)

            if response.status_code != 200:
                continue

            data = response.json()
            stat = {
                'player_points':   'PTS',
                'player_rebounds': 'REB',
                'player_assists':  'AST',
                'player_threes':   'FG3M',
            }[market]

            for bookmaker in data.get('bookmakers', []):
                for mkt in bookmaker.get('markets', []):
                    players_seen = {}
                    for outcome in mkt.get('outcomes', []):
                        player = outcome['description']
                        if player not in players_seen:
                            players_seen[player] = outcome['point']
                    for player, line in players_seen.items():
                        current_lines[(player, stat)] = line

    updated = 0
    for bet in log:
        if bet['date'] == today and bet['closing_line'] is None:
            key = (bet['player'], bet['stat'])
            if key in current_lines:
                bet['closing_line'] = current_lines[key]
                updated += 1

    save_bets_log(log)
    print(f"✅ Closing lines saved for {updated} bets")
    return current_lines


def auto_update_results(date=None):
    """
    Automatically pulls actual WNBA stats and updates bet results.
    Run 30-60 min after games finish.

    date: optional override — defaults to today
          use '2026-05-29' to update a previous day's bets
    """
    log         = load_bets_log()
    target_date = date if date else datetime.now().strftime('%Y-%m-%d')

    pending = [
        b for b in log
        if b['date'] == target_date and b['bet_result'] is None
    ]

    if len(pending) == 0:
        print(f"No pending bets to update for {target_date}.")
        print()
        print_performance_summary()
        return

    print(f"Updating {len(pending)} pending WNBA bets for {target_date}...")
    print()

    players_needed = list(set(b['player'] for b in pending))
    player_stats   = {}

    for player_name in players_needed:
        player_rows = df_master[df_master['PLAYER_NAME'] == player_name]

        if len(player_rows) == 0:
            print(f"  ⚠️  {player_name} not found in dataset")
            continue

        player_id = player_rows['Player_ID'].iloc[0]

        try:
            log_data = playergamelog.PlayerGameLog(
                player_id=player_id,
                season='2026',
                season_type_all_star='Regular Season',
                league_id_nullable='10'
            )
            time.sleep(1.5)

            df_log = log_data.get_data_frames()[0]

            if len(df_log) == 0:
                print(f"  ⚠️  No games found for {player_name}")
                continue

            df_log['GAME_DATE'] = pd.to_datetime(df_log['GAME_DATE'], format='mixed')
            latest    = df_log.sort_values('GAME_DATE').iloc[-1]
            game_date = latest['GAME_DATE'].strftime('%Y-%m-%d')

            player_stats[player_name] = {
                'PTS':  int(latest['PTS']),
                'REB':  int(latest['REB']),
                'AST':  int(latest['AST']),
                'FG3M': int(latest['FG3M']),
                'date': game_date,
            }

            print(f"  ✅ {player_name:<28} "
                  f"PTS:{latest['PTS']:.0f}  "
                  f"REB:{latest['REB']:.0f}  "
                  f"AST:{latest['AST']:.0f}  "
                  f"FG3M:{latest['FG3M']:.0f}  "
                  f"({game_date})")

        except Exception as e:
            print(f"  ❌ {player_name} — {e}")
            continue

    print()

    updated = 0
    skipped = 0

    for bet in log:
        if bet['date'] != target_date or bet['bet_result'] is not None:
            continue

        player = bet['player']
        stat   = bet['stat']

        if player not in player_stats:
            skipped += 1
            continue

        stats     = player_stats[player]
        game_date = stats['date']

        if game_date != target_date:
            print(f"  ⚠️  {player} — last game was {game_date}, not {target_date}")
            skipped += 1
            continue

        actual = stats[stat]

        if bet['recommendation'] == 'OVER':
            result = 'WIN' if actual > bet['line'] else (
                'PUSH' if actual == bet['line'] else 'LOSS'
            )
        else:
            result = 'WIN' if actual < bet['line'] else (
                'PUSH' if actual == bet['line'] else 'LOSS'
            )

        bet['actual_result'] = actual
        bet['bet_result']    = result

        if bet['closing_line'] is not None:
            if bet['recommendation'] == 'OVER':
                bet['clv'] = round(bet['line'] - bet['closing_line'], 1)
            else:
                bet['clv'] = round(bet['closing_line'] - bet['line'], 1)

        icon = '✅' if result == 'WIN' else '❌' if result == 'LOSS' else '➡️'
        print(f"  {icon} {player:<28} {stat:<6} "
              f"{bet['recommendation']} {bet['line']}  "
              f"Actual: {actual}  {result}")
        updated += 1

    save_bets_log(log)
    print()
    print(f"Updated: {updated}  Skipped: {skipped}")
    print()
    print_performance_summary()


print("✅ WNBA automated functions loaded")
print()

✅ WNBA automated functions loaded



In [76]:
# Run this 5 minutes before each tipoff window
# Change game_filter to match the teams playing in that window
# Only need to input one team from each matchup

# 1:00 PM game 6/6/2026:
save_closing_lines(df_bets, game_filter='Toronto Tempo')

# 3:00 PM game 6/6/2026
save_closing_lines(df_bets, game_filter='Las Vegas Aces')

# 6:00 PM game 6/6/2026
#save_closing_lines(df_bets, game_filter='Washington Mystics')

# 8:00 PM game 6/6/2026 
#save_closing_lines(df_bets, game_filter='Indiana Fever')
#save_closing_lines(df_bets, game_filter='New York Liberty')

save_closing_lines(df_bets, game_filter='Seattle Storm')

Filtering to games involving: Toronto Tempo
Games found: 1
✅ Closing lines saved for 24 bets
Filtering to games involving: Las Vegas Aces
Games found: 2
✅ Closing lines saved for 15 bets
Filtering to games involving: Seattle Storm
Games found: 1
✅ Closing lines saved for 10 bets


{('Sabrina Ionescu', 'PTS'): 17.5,
 ('Jonquel Jones', 'PTS'): 16.5,
 ('Dominique Malonga', 'PTS'): 16.5,
 ('Natisha Hiedeman', 'PTS'): 15.5,
 ("Flau'jae Johnson", 'PTS'): 11.5,
 ('Awa Fam', 'PTS'): 11.5,
 ('Leonie Fiebich', 'PTS'): 10.5,
 ('Marine Johannes', 'PTS'): 9.5,
 ('Pauline Astier', 'PTS'): 8.5,
 ('Zia Cooke', 'PTS'): 7.5,
 ('Jade Melbourne', 'PTS'): 5.5,
 ('Jonquel Jones', 'REB'): 9.5,
 ('Dominique Malonga', 'REB'): 8.5,
 ('Awa Fam', 'REB'): 6.5,
 ('Sabrina Ionescu', 'REB'): 5.5,
 ("Flau'jae Johnson", 'REB'): 4.5,
 ('Leonie Fiebich', 'REB'): 3.5,
 ('Natisha Hiedeman', 'REB'): 2.5,
 ('Natisha Hiedeman', 'AST'): 4.5,
 ('Sabrina Ionescu', 'AST'): 4.5,
 ('Pauline Astier', 'AST'): 3.5,
 ('Jade Melbourne', 'AST'): 3.5,
 ('Jonquel Jones', 'AST'): 2.5,
 ("Flau'jae Johnson", 'AST'): 2.5,
 ('Awa Fam', 'AST'): 2.5,
 ('Natisha Hiedeman', 'FG3M'): 2.5,
 ('Sabrina Ionescu', 'FG3M'): 2.5,
 ('Marine Johannes', 'FG3M'): 1.5,
 ('Jonquel Jones', 'FG3M'): 1.5,
 ('Leonie Fiebich', 'FG3M'): 1.5,
 (

In [11]:
# Safety reload — runs fast if already loaded, reloads if kernel died
import os

if 'df_master' not in dir() or 'model' not in dir():
    print("⚠️  Kernel was restarted — reloading...")
    %run "WNBA_Live_Pipeline.ipynb"  # re-runs the whole notebook
else:
    print("✅ Everything already loaded")

auto_update_results()

✅ Everything already loaded
No pending bets to update for 2026-07-02.

  WNBA PERFORMANCE SUMMARY
  Total bets logged:   446
  Completed:           427
  Pending:             16
  Voided:              3

  ── PLACED BETS (20%+ edge) ──────────────────
  Record:              34W - 26L - 0P
  Win rate:            56.7%

  By stat (placed only):
    PTS    22W/17L  (56%)  [39 bets]
    REB    4W/4L  (50%)  [8 bets]
    AST    6W/2L  (75%)  [8 bets]
    FG3M   2W/3L  (40%)  [5 bets]

  By confidence (placed only):
    🟢  23W/22L  (51%)  [45 bets]
    🟡  6W/2L  (75%)  [8 bets]
    🔴  5W/2L  (71%)  [7 bets]

  ── TRACKING ONLY (5-20% edge) ───────────────
  Record:              164W - 203L
  Win rate:            44.7%

  By stat (tracking only):
    PTS    56W/78L  (42%)  [134 bets]
    REB    47W/59L  (44%)  [106 bets]
    AST    40W/31L  (56%)  [71 bets]
    FG3M   21W/35L  (38%)  [56 bets]


In [21]:
# Run this 30-60 min after the final game ends
auto_update_results(date='2026-06-30')

Updating 13 pending WNBA bets for 2026-06-30...

  ✅ Breanna Stewart              PTS:15  REB:4  AST:0  FG3M:1  (2026-06-28)
  ✅ Jackie Young                 PTS:28  REB:5  AST:8  FG3M:4  (2026-06-28)
  ✅ Pauline Astier               PTS:5  REB:0  AST:4  FG3M:1  (2026-06-28)
  ✅ Leonie Fiebich               PTS:6  REB:0  AST:1  FG3M:0  (2026-06-28)
  ✅ Jonquel Jones                PTS:21  REB:7  AST:2  FG3M:3  (2026-06-28)
  ✅ NaLyssa Smith                PTS:6  REB:1  AST:2  FG3M:0  (2026-06-28)
  ✅ Chennedy Carter              PTS:11  REB:1  AST:1  FG3M:1  (2026-06-28)
  ✅ Marine Johannes              PTS:3  REB:1  AST:3  FG3M:1  (2026-06-28)
  ✅ Sabrina Ionescu              PTS:9  REB:3  AST:2  FG3M:2  (2026-06-28)

  ⚠️  Jackie Young — last game was 2026-06-28, not 2026-06-30
  ⚠️  Jonquel Jones — last game was 2026-06-28, not 2026-06-30
  ⚠️  Chennedy Carter — last game was 2026-06-28, not 2026-06-30
  ⚠️  Sabrina Ionescu — last game was 2026-06-28, not 2026-06-30
  ⚠️  NaLyssa Sm

In [112]:
# Retroactively tag all historical bets with bet_placed flag
log     = load_bets_log()
updated = 0

for bet in log:
    # Only update bets that don't already have the bet_placed field
    if 'bet_placed' not in bet:
        edge       = bet.get('edge', 0)
        confidence = bet.get('confidence', '🟢')

        # Apply same logic as current filter
        # Green tier (80+ games): 20%+ edge
        # Yellow tier (40-79 games): 20%+ edge
        # Red tier (<40 games): 25%+ edge
        if confidence == '🔴':
            bet['bet_placed'] = edge >= 25.0
        else:
            bet['bet_placed'] = edge >= 20.0

        updated += 1

save_bets_log(log)
print(f"Tagged {updated} historical bets")
print()

# Show breakdown
placed   = [b for b in log if b.get('bet_placed', False)]
tracking = [b for b in log if not b.get('bet_placed', True)]

print(f"Total bets:      {len(log)}")
print(f"Placed (20%+):   {len(placed)}")
print(f"Tracking (<20%): {len(tracking)}")
print()

# Quick check on placed vs tracking win rates for completed bets
completed = [b for b in log if b['bet_result'] is not None
             and b['bet_result'] != 'VOID']

placed_c   = [b for b in completed if b.get('bet_placed', False)]
tracking_c = [b for b in completed if not b.get('bet_placed', True)]

if len(placed_c) > 0:
    p_wins = len([b for b in placed_c if b['bet_result'] == 'WIN'])
    p_rate = p_wins / len(placed_c)
    print(f"Placed completed:    {len(placed_c)}  {p_wins}W  {p_rate:.1%}")

if len(tracking_c) > 0:
    t_wins = len([b for b in tracking_c if b['bet_result'] == 'WIN'])
    t_rate = t_wins / len(tracking_c)
    print(f"Tracking completed:  {len(tracking_c)}  {t_wins}W  {t_rate:.1%}")

Tagged 0 historical bets

Total bets:      433
Placed (20%+):   61
Tracking (<20%): 372

Placed completed:    60  34W  56.7%
Tracking completed:  367  164W  44.7%


In [114]:
# Re-run bias check with current predictions
# Run when refreshing bi weekly data so offsets stay accurate
print("Current systematic bias check:")
print()

for stat in ['PTS', 'REB', 'AST', 'FG3M']:
    stat_rows = df_results[df_results['stat'] == stat]
    if len(stat_rows) == 0:
        continue
    avg_raw_mu = stat_rows['mu_raw'].mean()
    avg_line   = stat_rows['line'].mean()
    gap        = avg_line - avg_raw_mu
    current_offset = CALIBRATION.get(stat, 0)
    net_gap    = gap - current_offset
    print(f"  {stat:<6}  Raw model: {avg_raw_mu:.1f}  "
          f"DK line: {avg_line:.1f}  "
          f"Gap: {gap:+.1f}  "
          f"Offset: {current_offset:+.1f}  "
          f"Over/under correcting: {net_gap:+.1f}")

Current systematic bias check:

  PTS     Raw model: 12.6  DK line: 12.4  Gap: -0.2  Offset: +1.3  Over/under correcting: -1.5
  REB     Raw model: 5.2  DK line: 5.3  Gap: +0.0  Offset: +0.2  Over/under correcting: -0.2
  AST     Raw model: 3.4  DK line: 3.7  Gap: +0.3  Offset: +0.3  Over/under correcting: -0.0
  FG3M    Raw model: 1.5  DK line: 1.4  Gap: -0.0  Offset: +0.1  Over/under correcting: -0.1


In [116]:
# Edge accuracy analysis
log       = load_bets_log()
completed = [b for b in log if b['bet_result'] is not None]

print(f"Total completed bets: {len(completed)}")
print()

# Bucket bets by edge range
buckets = {
    '5-10%':   [],
    '10-15%':  [],
    '15-20%':  [],
    '20-25%':  [],
    '25%+':    [],
}

for bet in completed:
    edge = bet['edge']
    if edge >= 25:
        buckets['25%+'].append(bet)
    elif edge >= 20:
        buckets['20-25%'].append(bet)
    elif edge >= 15:
        buckets['15-20%'].append(bet)
    elif edge >= 10:
        buckets['10-15%'].append(bet)
    elif edge >= 5:
        buckets['5-10%'].append(bet)

print(f"{'Edge Range':<12} {'Bets':>6} {'Wins':>6} {'Losses':>6} "
      f"{'Win %':>8} {'vs Breakeven':>14}")
print("-" * 55)

for bucket, bets in buckets.items():
    if len(bets) == 0:
        continue
    wins   = len([b for b in bets if b['bet_result'] == 'WIN'])
    losses = len([b for b in bets if b['bet_result'] == 'LOSS'])
    total  = wins + losses
    if total == 0:
        continue
    win_rate   = wins / total
    breakeven  = 0.524  # -110 juice
    vs_be      = win_rate - breakeven
    bar        = '█' * int(win_rate * 20)
    print(f"  {bucket:<10} {total:>6} {wins:>6} {losses:>6} "
          f"{win_rate:>8.1%} {vs_be:>+13.1%}  {bar}")

print()

# Also break down by stat and edge
print(f"{'Stat':<8} {'Edge 5-15%':>12} {'Edge 15%+':>12}")
print("-" * 35)

for stat in ['PTS', 'REB', 'AST', 'FG3M']:
    low_bets  = [b for b in completed
                 if b['stat'] == stat and 5 <= b['edge'] < 15]
    high_bets = [b for b in completed
                 if b['stat'] == stat and b['edge'] >= 15]

    def win_rate_str(bets):
        if len(bets) == 0:
            return 'N/A'
        wins = len([b for b in bets if b['bet_result'] == 'WIN'])
        return f"{wins/len(bets):.0%} ({len(bets)})"

    print(f"  {stat:<6}  {win_rate_str(low_bets):>12}  "
          f"{win_rate_str(high_bets):>12}")

print()

# Check if edge is predictive at all
# Correlation between edge and outcome
import numpy as np
edges    = [b['edge'] for b in completed]
outcomes = [1 if b['bet_result'] == 'WIN' else 0 for b in completed]
corr     = np.corrcoef(edges, outcomes)[0, 1]
print(f"Correlation between edge and outcome: {corr:+.3f}")
print()
if corr > 0.05:
    print("✅ Positive correlation — higher edge bets winning more often")
elif corr > -0.05:
    print("➡️  Near zero correlation — edge not yet predictive")
else:
    print("⚠️  Negative correlation — higher edge bets winning less often")

Total completed bets: 430

Edge Range     Bets   Wins Losses    Win %   vs Breakeven
-------------------------------------------------------
  5-10%         151     66     85    43.7%         -8.7%  ████████
  10-15%        110     52     58    47.3%         -5.1%  █████████
  15-20%         76     37     39    48.7%         -3.7%  █████████
  20-25%         49     22     27    44.9%         -7.5%  ████████
  25%+           41     21     20    51.2%         -1.2%  ██████████

Stat       Edge 5-15%    Edge 15%+
-----------------------------------
  PTS         42% (89)      48% (85)
  REB         45% (69)      43% (46)
  AST         54% (59)      70% (20)
  FG3M        41% (44)      28% (18)

Correlation between edge and outcome: +0.036

➡️  Near zero correlation — edge not yet predictive


In [118]:
#print_bias_trend(n_nights=3)

  BIAS CHECK TREND — last 3 nights

  PTS
  Date          Sample   Raw μ  DK Line    Gap  Offset  Remaining
  2026-06-24        38    12.8     13.0   +0.2    +1.3       -1.1
  2026-06-25        31    12.1     13.2   +1.2    +1.3       -0.1
  2026-06-26        27    12.6     12.4   -0.2    +1.3       -1.5
  → Average remaining gap:                      -0.92

  REB
  Date          Sample   Raw μ  DK Line    Gap  Offset  Remaining
  2026-06-24        34     4.9      5.0   +0.1    +0.2       -0.1
  2026-06-25        27     4.7      5.2   +0.5    +0.2       +0.3
  2026-06-26        22     5.2      5.3   +0.0    +0.2       -0.2
  → Average remaining gap:                      +0.01

  AST
  Date          Sample   Raw μ  DK Line    Gap  Offset  Remaining
  2026-06-24        18     4.2      4.4   +0.2    +0.3       -0.1
  2026-06-25        23     3.4      3.8   +0.4    +0.3       +0.1
  2026-06-26        13     3.4      3.6   +0.3    +0.3       -0.0
  → Average remaining gap:                  